In [1]:
%cd ..

/home/ubuntu/ScratchMLP-SingleChar


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import json

from src.model import MLP
from src.load_data import load_processed_data

In [3]:
data = load_processed_data()
X_train, Y_train, X_val, Y_val, X_test, Y_test = data

print(f"Data shapes:")
print(f"  Train: {X_train.shape[1]} samples")
print(f"  Validation: {X_val.shape[1]} samples")  
print(f"  Test: {X_test.shape[1]} samples")

Data shapes:
  Train: 101520 samples
  Validation: 11280 samples
  Test: 18800 samples


In [4]:
with open('models/best_params.json', 'r') as f:
        best_params = json.load(f)
        print(best_params)

n_hidden_layers = best_params['n_hidden_layers']
layer_dims = [784]
for i in range(n_hidden_layers):
    layer_dims.append(best_params[f'n_units_layer_{i}'])
layer_dims.append(47)

mlp = MLP(layer_dims=layer_dims, init='he', use_batchnorm=True)

weights = np.load('models/mlp_weights_tuned.npz')

loaded_params = {}
loaded_bn_params = {}

for key in weights.keys():
    if key.startswith('gamma') or key.startswith('beta') or key.startswith('running_'):
        loaded_bn_params[key] = weights[key]
    elif key.startswith('W') or key.startswith('b'):
        loaded_params[key] = weights[key]

mlp.parameters = loaded_params
if mlp.use_batchnorm:
    mlp.bn_params = loaded_bn_params

print(f"Loaded parameters: {list(loaded_params.keys())}")
if mlp.use_batchnorm:
    print(f"Loaded Batch Norm parameters: {list(loaded_bn_params.keys())}")

{'learning_rate': 0.0002830339354878112, 'lambda': 0.0348415168580236, 'keep_prob': 0.7, 'n_hidden_layers': 2, 'n_units_layer_0': 768, 'n_units_layer_1': 768}
Loaded parameters: ['W1', 'b1', 'W2', 'b2', 'W3', 'b3']
Loaded Batch Norm parameters: ['gamma1', 'beta1', 'running_mean1', 'running_var1', 'gamma2', 'beta2', 'running_mean2', 'running_var2']


In [5]:
def evaluate_model(mlp, X, Y, dataset_name):
    AL, _ = mlp.forward(X, is_training=False)
    predictions = np.argmax(AL, axis=0)
    true_labels = np.argmax(Y, axis=0)
    accuracy = np.mean(predictions == true_labels)
    loss = mlp.cost(AL, Y)
    
    print(f"{dataset_name} Results:")
    print(f"  Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Loss: {loss:.4f}")
    print(f"  Samples: {X.shape[1]}")
    print()
    
    return predictions, true_labels, accuracy, loss

train_preds, train_true, train_acc, train_loss = evaluate_model(mlp, X_train, Y_train, "Training")
val_preds, val_true, val_acc, val_loss = evaluate_model(mlp, X_val, Y_val, "Validation") 
test_preds, test_true, test_acc, test_loss = evaluate_model(mlp, X_test, Y_test, "Test")

Training Results:
  Accuracy: 0.9268 (92.68%)
  Loss: 0.1970
  Samples: 101520

Validation Results:
  Accuracy: 0.8776 (87.76%)
  Loss: 0.3650
  Samples: 11280

Test Results:
  Accuracy: 0.8675 (86.75%)
  Loss: 0.4036
  Samples: 18800

